# Exploratory Data Analysis

In this notebook we explore the **Walmart Recruiting: Store Sales
Forecasting** dataset. Our goal is to understand the data well enough that
we can make informed feature engineering and modeling choices in the next
notebook.

We will:

1. Load the three source files (`stores`, `features`, `train`) and inspect
   their shape, columns, and types.
2. Check data quality: missing values, duplicates, and value ranges.
3. Profile the **target** (`Weekly_Sales`): distribution, outliers, and
   time trend.
4. Look at structure across **stores** and **departments**.
5. Investigate the impact of **holidays**, **temperature**, **fuel price**,
   and **markdowns**.
6. Wrap up with a list of takeaways that will guide modeling choices.


## Setup

We will use **`pandas`** for tabular work, **`numpy`** for numeric helpers,
and **`plotly express`** for interactive visualizations. Plotly is well
suited to this dataset because the time series spans almost three years and
we will want to zoom and hover to inspect specific weeks.


In [1]:
import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "simple_white"

print("pandas version :", pd.__version__)
print("numpy version  :", np.__version__)
print("plotly version :", plotly.__version__)

pandas version : 3.1.1
numpy version  : 2.5.4
plotly version : 7.1.0


## Load the Data

The dataset is split across three CSV files. We load them straight from the
mirrored copy on GitHub so the notebook is reproducible without any local
downloads.


### `stores.csv`

One row per Walmart store (45 stores total). Tells us the **type** of each
store (`A`, `B`, or `C`, which loosely correspond to large / medium /
small format) and its **size** in square feet.


In [2]:
df_stores = pd.read_csv(
    "https://raw.githubusercontent.com/bdi593/datasets/main/walmart-store-sales-forecasting/stores.csv"
)
df_stores

,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875
5,6,A,202505
6,7,B,70713
7,8,A,155078
8,9,B,125833
9,10,B,126512


### `features.csv`

One row per `(Store, Week)`. Contains contextual variables Walmart thinks
might explain sales: weather, fuel price, the consumer price index,
unemployment rate, an `IsHoliday` flag, and five anonymized markdown
columns capturing promotional spend.

:::{note} Heads-up

The `MarkDown1` to `MarkDown5` columns are only available from November
2011 onward and are missing (`NaN`) before that. We will deal with this
when we engineer features.

:::

In [3]:
df_features = pd.read_csv(
    "https://raw.githubusercontent.com/bdi593/datasets/main/walmart-store-sales-forecasting/features.csv",
    parse_dates=["Date"],
)
df_features.head()

,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,2010-02-05,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,2010-02-12,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True
2,1,2010-02-19,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False
3,1,2010-02-26,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False
4,1,2010-03-05,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False


### `train.csv`

One row per `(Store, Department, Week)`; this is our **target series**.
The column `Weekly_Sales` is what we will eventually try to forecast.


In [4]:
df_train = pd.read_csv(
    "https://raw.githubusercontent.com/bdi593/datasets/main/walmart-store-sales-forecasting/train.csv",
    parse_dates=["Date"],
)
df_train.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,2010-02-05,24924.50,False
1,1,1,2010-02-12,46039.49,True
2,1,1,2010-02-19,41595.55,False
3,1,1,2010-02-26,19403.54,False
4,1,1,2010-03-05,21827.90,False


## Data Shape and Types

A quick sanity check: how big is each table, what types are the columns,
and how much of the data is missing?


In [5]:
for name, df in [("stores", df_stores), ("features", df_features), ("train", df_train)]:
    print(f"--- {name} ---")
    print(f"  shape : {df.shape}")
    print(
        f"  date range: {df['Date'].min()} → {df['Date'].max()}"
        if "Date" in df.columns
        else ""
    )
    print(df.dtypes)
    print()

--- stores ---
  shape : (45, 3)

Store    int64
Type       str
Size     int64
dtype: object

--- features ---
  shape : (8190, 12)
  date range: 2010-02-05 00:00:00 → 2013-07-26 00:00:00
Store                    int64
Date            datetime64[us]
Temperature            float64
Fuel_Price             float64
MarkDown1              float64
MarkDown2              float64
MarkDown3              float64
MarkDown4              float64
MarkDown5              float64
CPI                    float64
Unemployment           float64
IsHoliday                 bool
dtype: object

--- train ---
  shape : (421570, 5)
  date range: 2010-02-05 00:00:00 → 2012-10-26 00:00:00
Store                    int64
Dept                     int64
Date            datetime64[us]
Weekly_Sales           float64
IsHoliday                 bool
dtype: object



### Missing Values

We expect the `MarkDown` columns in `features` to be largely missing
(they only start in late 2011). Anything else missing would be a surprise.


In [6]:
df_features.isna().mean().sort_values(ascending=False).round(3)

MarkDown2       0.643
MarkDown4       0.577
MarkDown3       0.559
MarkDown1       0.508
MarkDown5       0.505
CPI             0.071
Unemployment    0.071
Store           0.000
Fuel_Price      0.000
Temperature     0.000
Date            0.000
IsHoliday       0.000
dtype: float64

In [7]:
print("missing in stores :", df_stores.isna().sum().sum())
print("missing in train  :", df_train.isna().sum().sum())

missing in stores : 0
missing in train  : 0


As expected, the markdown columns are missing for the majority of weeks.
The `CPI` and `Unemployment` columns are missing only at the very end of
`features.csv` (from May 2013), after the last week of sales in
`train.csv`, so they do not affect the weeks we model. The training table
itself has no missing values.

## Stores

Let's profile the 45 stores by **type** and **size** before diving into
sales.


In [8]:
df_stores["Type"].value_counts()

Type
A    22
B    17
C     6
Name: count, dtype: int64

In [9]:
fig = px.box(
    df_stores,
    x="Type",
    y="Size",
    color="Type",
    points="all",
    title="Store size (sq ft) by store type",
    template="simple_white",
)
fig.show()

**Takeaway.** Type `A` stores are the largest, type `C` are the smallest,
and type `B` overlaps the middle. Store _size_ is not just a label. It is
a strong predictor of how much a store can sell, and we will keep both the
type and the raw size as features.


## Target Variable: `Weekly_Sales`

The thing we ultimately want to forecast. We will look at:

- the **distribution** of weekly sales across all `(Store, Dept, Week)`
  rows,
- whether there are any **negative or zero** sales (returns or closed
  departments),
- and the overall **time trend** at the chain level.


In [10]:
df_train["Weekly_Sales"].describe()

count    421570.000000
mean      15981.258123
std       22711.183519
min       -4988.940000
25%        2079.650000
50%        7612.030000
75%       20205.852500
max      693099.360000
Name: Weekly_Sales, dtype: float64

A handful of rows have **negative** weekly sales (weeks where returns
exceeded purchases for that department). We will leave them in for now
but flag them as something to watch.

In [11]:
neg = (df_train["Weekly_Sales"] < 0).sum()
print(f"rows with negative weekly sales: {neg:,} ({neg / len(df_train):.2%})")

rows with negative weekly sales: 1,285 (0.30%)


:::{tip} Aggregate before you plot large data

Plotly stores every input value inside the figure. A histogram or box plot
of all 421,570 rows would add several megabytes to the page and slow down
the browser. Instead we let `numpy` count the rows in each bin and plot the
counts with `px.bar`. The chart looks the same, but the figure holds only
80 numbers.

:::

In [12]:
# Count rows per bin first, then plot the 80 bin counts.
counts, edges = np.histogram(df_train["Weekly_Sales"], bins=80)
sales_hist = pd.DataFrame(
    {"Weekly_Sales": (edges[:-1] + edges[1:]) / 2, "Rows": counts}
)

fig = px.bar(
    sales_hist,
    x="Weekly_Sales",
    y="Rows",
    title="Distribution of Weekly_Sales (raw)",
    template="simple_white",
)
fig.update_traces(width=edges[1] - edges[0])
fig.update_layout(bargap=0)
fig.show()

The distribution is **highly right-skewed**: about three quarters of
department-weeks sit under \$20k, while a thin tail stretches to almost
\$700k. A log transform gives a much more readable picture and is the kind
of trick we may want to apply when training certain models.

In [13]:
positive = df_train.loc[df_train["Weekly_Sales"] > 0, "Weekly_Sales"]
counts, edges = np.histogram(np.log1p(positive), bins=60)
log_hist = pd.DataFrame({"log_sales": (edges[:-1] + edges[1:]) / 2, "Rows": counts})

fig = px.bar(
    log_hist,
    x="log_sales",
    y="Rows",
    title="Distribution of log(Weekly_Sales + 1), positive sales only",
    template="simple_white",
)
fig.update_traces(width=edges[1] - edges[0])
fig.update_layout(bargap=0)
fig.show()

### Total Weekly Sales over Time

Aggregating up to the **chain level** (sum over all stores and
departments) shows the overall demand pattern.


In [14]:
weekly_total = df_train.groupby("Date", as_index=False)["Weekly_Sales"].sum()

fig = px.line(
    weekly_total,
    x="Date",
    y="Weekly_Sales",
    title="Total Walmart weekly sales (all stores & departments)",
    template="simple_white",
)
fig.show()

**Takeaway.** Two things jump out:

1. **Strong yearly seasonality.** Every November/December has a massive
   spike. The two biggest weeks are the weeks just before Christmas,
   followed by the Thanksgiving / Black Friday weeks.
2. **Weak overall trend.** Outside of the holiday spikes the level is
   fairly flat across the three years.

Any forecasting model we build _must_ capture this seasonality, either via
calendar features, lagged values, or a seasonal model component.

## Holidays

The dataset highlights four "official" holidays: **Super Bowl**, **Labor
Day**, **Thanksgiving**, and **Christmas**. The Kaggle competition
weights errors on holiday weeks 5× higher than non-holiday weeks, so it
pays to look at them carefully.


In [15]:
holiday_summary = (
    df_train.groupby("IsHoliday")["Weekly_Sales"]
    .agg(["count", "mean", "median"])
    .round(2)
)
holiday_summary

,count,mean,median
IsHoliday,,,
False,391909,15901.45,7589.95
True,29661,17035.82,7947.74


The four holidays are not alike. Using the holiday weeks listed with the
dataset, we can split the holiday rows by holiday:

In [16]:
holiday_weeks = {
    "Super Bowl": ["2010-02-12", "2011-02-11", "2012-02-10"],
    "Labor Day": ["2010-09-10", "2011-09-09", "2012-09-07"],
    "Thanksgiving": ["2010-11-26", "2011-11-25"],
    "Christmas": ["2010-12-31", "2011-12-30"],
}

holiday_name = pd.Series("Not a holiday", index=df_train.index, name="Holiday")
for name, dates in holiday_weeks.items():
    holiday_name.loc[df_train["Date"].isin(pd.to_datetime(dates))] = name

(
    df_train.groupby(holiday_name)["Weekly_Sales"]
    .agg(["count", "mean", "median"])
    .round(2)
    .sort_values("mean", ascending=False)
)

,count,mean,median
Holiday,,,
Thanksgiving,5959,22220.94,10007.38
Super Bowl,8895,16378.00,7969.23
Not a holiday,391909,15901.45,7589.95
Labor Day,8861,15881.69,7502.66
Christmas,5946,14543.39,7046.26


The box plot below summarizes more than 400,000 rows, so we compute the
quartiles and whiskers first and draw the boxes from those numbers (see
the tip above). Points beyond the whiskers are not drawn.

In [17]:
def box_stats(values):
    # Quartiles and Tukey whiskers (furthest points within 1.5 IQR).
    q1, median, q3 = values.quantile([0.25, 0.5, 0.75])
    iqr = q3 - q1
    return {
        "q1": q1,
        "median": median,
        "q3": q3,
        "lowerfence": values[values >= q1 - 1.5 * iqr].min(),
        "upperfence": values[values <= q3 + 1.5 * iqr].max(),
    }


holiday_box = pd.DataFrame(
    {flag: box_stats(group) for flag, group in df_train.groupby("IsHoliday")["Weekly_Sales"]}
).T

fig = go.Figure(
    go.Box(
        x=holiday_box.index.astype(str),
        q1=holiday_box["q1"],
        median=holiday_box["median"],
        q3=holiday_box["q3"],
        lowerfence=holiday_box["lowerfence"],
        upperfence=holiday_box["upperfence"],
        name="Weekly_Sales",
    )
)
fig.update_layout(
    title="Weekly_Sales: holiday vs. non-holiday weeks (outliers not drawn)",
    xaxis_title="IsHoliday",
    yaxis_title="Weekly_Sales",
)
fig.show()

**Takeaway.** Taken together, holiday weeks have a higher mean than other
weeks (about 7% higher), but the median and the box barely move. The
table by holiday shows why: the Thanksgiving week sells far more than an
ordinary week, the Super Bowl week a little more, Labor Day about the
same, and the week flagged as Christmas _less_. That flagged week is the
last week of December, after the peak shopping weeks; the chain-level
chart above shows the real peak the week before.

A single `IsHoliday` flag mixes these very different effects, so labeling
_which_ holiday it is (and the weeks around it) is a useful feature
engineering idea for the modeling notebook.

## Store and Department Breakdown

The chain-level view hides a lot of variation. Walmart sells very
different mixes of products in different departments, and store size
affects total volume.


In [18]:
sales_by_store = (
    df_train.groupby("Store", as_index=False)["Weekly_Sales"]
    .sum()
    .merge(df_stores, on="Store")
    .sort_values("Weekly_Sales", ascending=False)
)

fig = px.bar(
    sales_by_store,
    x="Store",
    y="Weekly_Sales",
    color="Type",
    title="Total sales by store, colored by store type",
    template="simple_white",
)
fig.show()

As expected, large type-`A` stores dominate total sales: nine of the ten
highest-selling stores are type `A`. Store _type_ will likely be a useful
categorical feature.

In [19]:
sales_by_dept = (
    df_train.groupby("Dept", as_index=False)["Weekly_Sales"]
    .sum()
    .sort_values("Weekly_Sales", ascending=False)
    .head(20)
)

fig = px.bar(
    sales_by_dept,
    x="Dept",
    y="Weekly_Sales",
    title="Top 20 departments by total sales",
    template="simple_white",
)
fig.update_xaxes(type="category")
fig.show()

A handful of departments account for a large share of revenue: these top
20 of the 81 departments bring in about two thirds of total sales. For
modeling we will want to forecast at the `(Store, Dept)` level rather than
the store level.

## Joining Sales with Features

To examine how weather, fuel price, and markdowns relate to sales we join
`train` with `features` on `(Store, Date)`.


In [20]:
df = df_train.merge(df_features, on=["Store", "Date", "IsHoliday"], how="left").merge(
    df_stores, on="Store", how="left"
)
print("merged shape:", df.shape)
df.head()

merged shape: (421570, 16)


,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size
0,1,1,2010-02-05,24924.50,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,A,151315
1,1,1,2010-02-12,46039.49,True,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,A,151315
2,1,1,2010-02-19,41595.55,False,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,A,151315
3,1,1,2010-02-26,19403.54,False,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,A,151315
4,1,1,2010-03-05,21827.90,False,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,A,151315


### Temperature vs. Sales

Average regional temperature could matter (think: heaters, snow shovels,
swimsuits). We aggregate to the weekly chain level so the picture isn't
dominated by store-level noise.


In [21]:
weekly = (
    df.groupby("Date")
    .agg(
        Weekly_Sales=("Weekly_Sales", "sum"),
        Temperature=("Temperature", "mean"),
        Fuel_Price=("Fuel_Price", "mean"),
        CPI=("CPI", "mean"),
        Unemployment=("Unemployment", "mean"),
    )
    .reset_index()
)

fig = px.scatter(
    weekly,
    x="Temperature",
    y="Weekly_Sales",
    trendline="ols",
    title="Avg. weekly temperature vs. total weekly sales",
    template="simple_white",
)
fig.show()

**Takeaway.** The relationship is weak at the chain level. Total sales
are similar across most temperature ranges, with the biggest spikes
driven by _holidays_, not weather. At the **department** level the
relationship would likely be much stronger (e.g., garden vs. winter
apparel), but capturing that is a job for an interaction feature in the
model, not a single global slope.


### Fuel Price and Unemployment over Time

Both rise and fall with the broader economy. They are easy features to
include but are unlikely to _cause_ short-term swings in weekly sales;
they are slow-moving context, not week-to-week shocks.


In [22]:
fig = px.line(
    weekly.melt(id_vars="Date", value_vars=["Fuel_Price", "Unemployment", "CPI"]),
    x="Date",
    y="value",
    color="variable",
    facet_row="variable",
    title="Macro context over time",
    height=600,
    template="simple_white",
)
fig.update_yaxes(matches=None)
fig.show()

### Markdowns

Markdowns are promotional spend by category. They are missing for every
week before 11 November 2011 and widely populated after that. Markdowns
are recorded per store and week, so we total them from `features.csv`
over the weeks covered by `train.csv`.

In [23]:
markdown_cols = [f"MarkDown{i}" for i in range(1, 6)]

md_long = (
    df_features[df_features["Date"] <= df_train["Date"].max()]
    .groupby("Date")[markdown_cols]
    .sum(min_count=1)
    .reset_index()
    .melt(id_vars="Date", var_name="MarkDown", value_name="Spend")
)

fig = px.line(
    md_long,
    x="Date",
    y="Spend",
    color="MarkDown",
    title="Total markdown spend by week (all stores)",
    template="simple_white",
)
fig.show()

**Takeaway.** Markdown spend is concentrated in a few weeks. The largest
weekly total is in early February 2012, the week before the Super Bowl,
followed by Thanksgiving week 2011 and the last week of December 2011.
The five markdown columns peak in different weeks. They could be useful
features, but their **missingness pattern** means we will need to be
careful when training a model that mixes weeks before and after November
2011.

## A Single Series Up Close

Looking at one `(Store, Dept)` series gives us intuition for the
forecasting problem we are about to tackle.


In [24]:
one_series = df_train.query("Store == 1 and Dept == 1").sort_values("Date")

fig = px.line(
    one_series,
    x="Date",
    y="Weekly_Sales",
    title="Store 1, Department 1: weekly sales",
    template="simple_white",
)
fig.show()

Even in a single department you can see clear yearly seasonality and a
few sharp spikes. Some line up with the flagged holidays, but the two
largest fall in early April, in the weeks just before Easter, which is not
one of the four flagged holidays. A good forecaster will need to learn
both the **level** of each `(Store, Dept)` series and its **seasonal
pattern**.

## Key Takeaways

Things to carry into the modeling notebook:

1. **Forecast at the `(Store, Dept)` grain.** That is the natural unit of
   the data and where the variation lives.
2. **Seasonality is huge.** Calendar features (week of year, month,
   holiday flags, weeks-to-next-holiday) will be essential.
3. **Holidays drive the biggest swings.** Encoding _which_ holiday is
   coming up matters more than just whether the week is a holiday.
4. **Store type and size** are useful static features.
5. **Markdown columns are missing before November 2011**, about two
   thirds of the sales weeks. Handle their `NaN`s explicitly (impute,
   indicator flag, or restrict to post-2011 training).
6. **Weekly sales are right-skewed and occasionally negative.** Consider a
   log or `log1p` transform for linear-style models, and decide how to
   treat the negative-sales rows (drop, clip to 0, or model directly).
7. **Use a time-based train/test split.** Random shuffles will leak
   future weeks into the training set.